### Импорты:

In [41]:
import numpy as np
from collections import Counter

from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.tree import DecisionTreeClassifier
import time

###  Вспомогательные функции

In [42]:
def gini(y) -> float:
    """Индекс Джини: 1 - sum(p_k^2)"""
    if len(y) == 0:
        return 0.0
    total = len(y)
    return 1.0 - sum((c / total) ** 2 for c in Counter(y).values())


def entropy(y) -> float:
    """Энтропия Шеннона: -sum(p_k * log2(p_k))"""
    if len(y) == 0:
        return 0.0
    total = len(y)
    result = 0.0
    for c in Counter(y).values():
        p = c / total
        if p > 0:
            result -= p * np.log2(p)
    return result


CRITERIA = {"gini": gini, "entropy": entropy}

### Реализация дерева решений:

In [43]:
def make_node(feature=None, threshold=None,
              left=None, right=None, value=None) -> dict:
    """Создаёт узел дерева как словарь."""
    return {
        "feature":   feature,    # индекс признака для сплита
        "threshold": threshold,  # порог: левый ребёнок <= threshold
        "left":      left,       # левое поддерево
        "right":     right,      # правое поддерево
        "value":     value,      # класс-большинство (только в листьях)
    }

def is_leaf(node: dict) -> bool:
    return node["value"] is not None

Перебираем все признаки и пороги (средние между соседними уникальными значениями).  
Выбираем то, что максимизирует прирост информации.

In [44]:
def best_split(X, y, impurity_fn) -> tuple:
    """
    Возвращает (feature_index, threshold) с максимальным приростом.
    Если улучшений нет — (None, None).
    """
    best_gain, best_feat, best_thresh = -np.inf, None, None
    parent_imp = impurity_fn(y)

    for feat in range(X.shape[1]):
        values = np.unique(X[:, feat])
        if len(values) < 2:
            continue
        # пороги — середины между соседними уникальными значениями
        for thresh in (values[:-1] + values[1:]) / 2:
            mask = X[:, feat] <= thresh
            nl, nr = mask.sum(), (~mask).sum()
            if nl == 0 or nr == 0:
                continue
            gain = parent_imp - (
                nl / len(y) * impurity_fn(y[mask]) +
                nr / len(y) * impurity_fn(y[~mask])
            )
            if gain > best_gain:
                best_gain, best_feat, best_thresh = gain, feat, thresh

    return best_feat, best_thresh

Построение дерева: `grow` и `fit`

In [45]:
def grow(X, y, impurity_fn, max_depth, min_samples_split,
         min_samples_leaf, depth=0) -> dict:
    """Рекурсивно строит дерево. Возвращает корневой узел."""

    majority = Counter(y).most_common(1)[0][0]

    # --- условия остановки ---
    if (len(np.unique(y)) == 1                              # узел чистый
            or len(y) < min_samples_split                   # мало образцов
            or (max_depth is not None and depth >= max_depth)):  # достигли глубины
        return make_node(value=majority)

    feat, thresh = best_split(X, y, impurity_fn)
    if feat is None:                                        # нет полезного сплита
        return make_node(value=majority)

    mask = X[:, feat] <= thresh
    if mask.sum() < min_samples_leaf or (~mask).sum() < min_samples_leaf:
        return make_node(value=majority)

    return make_node(
        feature=feat,
        threshold=thresh,
        left=grow(X[mask],  y[mask],  impurity_fn, max_depth,
                  min_samples_split, min_samples_leaf, depth + 1),
        right=grow(X[~mask], y[~mask], impurity_fn, max_depth,
                   min_samples_split, min_samples_leaf, depth + 1),
    )


def fit(X, y, *, max_depth=None, min_samples_split=2,
        min_samples_leaf=1, criterion="gini") -> dict:
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)
    return grow(X, y, CRITERIA[criterion],
                max_depth, min_samples_split, min_samples_leaf)

Доп

In [46]:
def tree_depth(node: dict) -> int:
    """Максимальная глубина дерева."""
    if is_leaf(node):
        return 0
    return 1 + max(tree_depth(node["left"]), tree_depth(node["right"]))


def count_nodes(node: dict) -> int:
    """Общее количество узлов."""
    if is_leaf(node):
        return 1
    return 1 + count_nodes(node["left"]) + count_nodes(node["right"])

Предсказание

In [47]:
def traverse(node: dict, x) -> int:
    """Проходит по дереву для одного образца."""
    if is_leaf(node):
        return node["value"]
    if x[node["feature"]] <= node["threshold"]:
        return traverse(node["left"], x)
    return traverse(node["right"], x)


def predict(root: dict, X) -> np.ndarray:
    """Предсказывает классы для всех образцов."""
    return np.array([traverse(root, x) for x in X])

### Подготовка данных и сравнение:

In [ ]:
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target,
    test_size=0.25, random_state=42, stratify=iris.target
)

# --- кастомное дерево ---
t0 = time.perf_counter()
root = fit(X_train, y_train, max_depth=4, criterion="entropy")
fit_time_custom = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred = predict(root, X_test)
pred_time_custom = time.perf_counter() - t0

# --- sklearn ---
t0 = time.perf_counter()
sk = DecisionTreeClassifier(max_depth=4, criterion="entropy", random_state=42)
sk.fit(X_train, y_train)
fit_time_sk = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_sk = sk.predict(X_test)
pred_time_sk = time.perf_counter() - t0

acc    = accuracy_score(y_test, y_pred)
acc_sk = accuracy_score(y_test, y_pred_sk)

print(f"{'Метрика':<22} {'Кастомное':>12} {'sklearn':>12}")
print("─" * 46)
print(f"{'Accuracy':<22} {acc:>12.4f} {acc_sk:>12.4f}")
print(f"{'Время fit (с)':<22} {fit_time_custom:>12.6f} {fit_time_sk:>12.6f}")
print(f"{'Время predict (с)':<22} {pred_time_custom:>12.6f} {pred_time_sk:>12.6f}")
print(f"{'Глубина':<22} {tree_depth(root):>12} {sk.get_depth():>12}")
print(f"{'Узлов':<22} {count_nodes(root):>12} {sk.tree_.node_count:>12}")

print()
print(classification_report(y_test, y_pred, target_names=iris.target_names))

Метрика                   Кастомное      sklearn
──────────────────────────────────────────────
Accuracy                     0.9211       0.9211
Время fit (с)              0.006292     0.000706
Время predict (с)          0.000062     0.000638
Глубина                           4            4
Узлов                            13           13

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        12
  versicolor       0.86      0.92      0.89        13
   virginica       0.92      0.85      0.88        13

    accuracy                           0.92        38
   macro avg       0.92      0.92      0.92        38
weighted avg       0.92      0.92      0.92        38

